# Time Series Analysis and Forecasting of Electricity Demand in Kenya

# Data Understanding and Cleaning

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Load the datasets
demand_df = pd.read_csv("../data/kenya-electricity-demand.csv")

generation_df = pd.read_csv("../data/kenya-electricity-generation.csv")

percapita_df = pd.read_csv("../data/kenya-per-capita-electricity-generation.csv")

population_df = pd.read_csv("../data/kenya-population.csv")

In [4]:
# Preview the datasets
print("Electricity Demand Dataset")
display(demand_df.head())

print("Electricity Generation Dataset")
display(generation_df.head())

print("Per Capita Generation Dataset")
display(percapita_df.head())

print("Population Dataset")
display(population_df.head())

Electricity Demand Dataset


,Country,Code,Year,Electricity-demand
0,Kenya,KEN,2000,4.51
1,Kenya,KEN,2001,4.98
2,Kenya,KEN,2002,5.37
3,Kenya,KEN,2003,5.67
4,Kenya,KEN,2004,6.32


Electricity Generation Dataset


,Country,Code,Year,Coal,Fossil-fuels,Gas,Hydro,Low-carbon,Nuclear,Oil,Renewables,Solar,Solar-wind,Wind,Geothermal,Total-electricity-produced
0,Kenya,KEN,2000,0.0,2.57,0.0,1.31,1.74,0.0,2.57,1.74,0.0,0.0,0.0,0.43,4.31
1,Kenya,KEN,2001,0.0,1.95,0.0,2.38,2.86,0.0,1.95,2.86,0.0,0.0,0.0,0.48,4.81
2,Kenya,KEN,2002,0.0,1.20,0.0,3.09,3.95,0.0,1.20,3.95,0.0,0.0,0.0,0.86,5.15
3,Kenya,KEN,2003,0.0,0.99,0.0,3.23,4.49,0.0,0.99,4.49,0.0,0.0,0.0,1.26,5.48
4,Kenya,KEN,2004,0.0,1.82,0.0,2.84,4.34,0.0,1.82,4.34,0.0,0.0,0.0,1.50,6.16


Per Capita Generation Dataset


,Country,Code,Year,Coal-per-capita,Fossil-fuels-per-capita,Gas-per-capita,Hydro-per-capita,Low-carbon-per-capita,Nuclear-per-capita,Oil-per-capita,Renewable-per-capita,Solar-per-capita,Solar-wind-per-capita,Wind-per-capita,Geothermal-per-capita,Total-electricity-per-capita-produced
0,Kenya,KEN,2000,0.0,83.869360,0.0,42.750530,56.783150,0.0,83.869360,56.783150,0.0,0.0,0.0,14.03262,140.652510
1,Kenya,KEN,2001,0.0,61.671448,0.0,75.270800,90.451454,0.0,61.671448,90.451454,0.0,0.0,0.0,15.18065,152.122902
2,Kenya,KEN,2002,0.0,36.776188,0.0,94.698685,121.054955,0.0,36.776188,121.054955,0.0,0.0,0.0,26.35627,157.831143
3,Kenya,KEN,2003,0.0,29.418552,0.0,95.981740,133.423540,0.0,29.418552,133.423540,0.0,0.0,0.0,37.44180,162.842092
4,Kenya,KEN,2004,0.0,52.429240,0.0,81.812660,125.023580,0.0,52.429240,125.023580,0.0,0.0,0.0,43.21092,177.452820


Population Dataset


,Country,Year,Population
0,Kenya,2000,30642894
1,Kenya,2001,31619171
2,Kenya,2002,32629809
3,Kenya,2003,33652233
4,Kenya,2004,34713452


In [5]:
# Check dataset shapes
print("Demand Shape:", demand_df.shape)
print("Generation Shape:", generation_df.shape)
print("Per Capita Shape:", percapita_df.shape)
print("Population Shape:", population_df.shape)

Demand Shape: (25, 4)
Generation Shape: (25, 16)
Per Capita Shape: (25, 16)
Population Shape: (25, 3)


In [6]:
# Checking dataset info
print("Demand Dataset Info")
demand_df.info()


Demand Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country             25 non-null     object 
 1   Code                25 non-null     object 
 2   Year                25 non-null     int64  
 3   Electricity-demand  25 non-null     float64
dtypes: float64(1), int64(1), object(2)
memory usage: 932.0+ bytes


In [7]:
generation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Country                     25 non-null     object 
 1   Code                        25 non-null     object 
 2   Year                        25 non-null     int64  
 3   Coal                        25 non-null     float64
 4   Fossil-fuels                25 non-null     float64
 5   Gas                         25 non-null     float64
 6   Hydro                       25 non-null     float64
 7   Low-carbon                  25 non-null     float64
 8   Nuclear                     25 non-null     float64
 9   Oil                         25 non-null     float64
 10  Renewables                  25 non-null     float64
 11  Solar                       25 non-null     float64
 12  Solar-wind                  25 non-null     float64
 13  Wind                        25 non-nu

In [8]:
percapita_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 16 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Country                                25 non-null     object 
 1   Code                                   25 non-null     object 
 2   Year                                   25 non-null     int64  
 3   Coal-per-capita                        25 non-null     float64
 4   Fossil-fuels-per-capita                25 non-null     float64
 5   Gas-per-capita                         25 non-null     float64
 6   Hydro-per-capita                       25 non-null     float64
 7   Low-carbon-per-capita                  25 non-null     float64
 8   Nuclear-per-capita                     25 non-null     float64
 9   Oil-per-capita                         25 non-null     float64
 10  Renewable-per-capita                   25 non-null     float64
 11  Solar-pe

In [9]:
population_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Country     25 non-null     object
 1   Year        25 non-null     int64 
 2   Population  25 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 732.0+ bytes


In [11]:
print("demand_df:", demand_df.isnull().sum().sum())
print("generation_df:", generation_df.isnull().sum().sum())
print("percapita_df:", percapita_df.isnull().sum().sum())
print("population_df:", population_df.isnull().sum().sum())

demand_df: 0
generation_df: 0
percapita_df: 0
population_df: 0


There is no missing values in the datasets

In [12]:
# Checking for duplicates
print("demand_df:", demand_df.duplicated().sum())
print("generation_df:", generation_df.duplicated().sum())   
print("percapita_df:", percapita_df.duplicated().sum())
print("population_df:", population_df.duplicated().sum())


demand_df: 0
generation_df: 0
percapita_df: 0
population_df: 0


There is no duplicates in the datasets

In [14]:
# Check column names
print("Demand Columns:", demand_df.columns) 
print("Generation Columns:", generation_df.columns)
print("Per Capita Columns:", percapita_df.columns)
print("Population Columns:", population_df.columns)

Demand Columns: Index(['Country', 'Code', 'Year', 'Electricity-demand'], dtype='object')
Generation Columns: Index(['Country', 'Code', 'Year', 'Coal', 'Fossil-fuels', 'Gas', 'Hydro',
       'Low-carbon', 'Nuclear', 'Oil', 'Renewables', 'Solar', 'Solar-wind',
       'Wind', 'Geothermal', 'Total-electricity-produced'],
      dtype='object')
Per Capita Columns: Index(['Country', 'Code', 'Year', 'Coal-per-capita', 'Fossil-fuels-per-capita',
       'Gas-per-capita', 'Hydro-per-capita', 'Low-carbon-per-capita',
       'Nuclear-per-capita', 'Oil-per-capita', 'Renewable-per-capita',
       'Solar-per-capita', 'Solar-wind-per-capita', 'Wind-per-capita',
       'Geothermal-per-capita', 'Total-electricity-per-capita-produced'],
      dtype='object')
Population Columns: Index(['Country', 'Year', 'Population'], dtype='object')


In [17]:
# Check data types
print("Demand Data Types:", demand_df.dtypes)
print("Generation Data Types:", generation_df.dtypes)
print("Per Capita Data Types:", percapita_df.dtypes)
print("Population Data Types:", population_df.dtypes)

Demand Data Types: Country                object
Code                   object
Year                    int64
Electricity-demand    float64
dtype: object
Generation Data Types: Country                        object
Code                           object
Year                            int64
Coal                          float64
Fossil-fuels                  float64
Gas                           float64
Hydro                         float64
Low-carbon                    float64
Nuclear                       float64
Oil                           float64
Renewables                    float64
Solar                         float64
Solar-wind                    float64
Wind                          float64
Geothermal                    float64
Total-electricity-produced    float64
dtype: object
Per Capita Data Types: Country                                   object
Code                                      object
Year                                       int64
Coal-per-capita                 

In [18]:
# Summary statistics
print("Demand Summary Statistics")
print(demand_df.describe())
print("Generation Summary Statistics")
print(generation_df.describe())
print("Per Capita Summary Statistics")
print(percapita_df.describe())
print("Population Summary Statistics")
print(population_df.describe())


Demand Summary Statistics
              Year  Electricity-demand
count    25.000000           25.000000
mean   2012.000000            8.767200
std       7.359801            2.866616
min    2000.000000            4.510000
25%    2006.000000            6.560000
50%    2012.000000            8.080000
75%    2018.000000           11.390000
max    2024.000000           13.730000
Generation Summary Statistics
              Year  Coal  Fossil-fuels   Gas      Hydro  Low-carbon  Nuclear  \
count    25.000000  25.0     25.000000  25.0  25.000000    25.00000     25.0   
mean   2012.000000   0.0      1.697600   0.0   3.216800     6.93320      0.0   
std       7.359801   0.0      0.560493   0.0   0.650184     3.07592      0.0   
min    2000.000000   0.0      0.750000   0.0   1.310000     1.74000      0.0   
25%    2006.000000   0.0      1.260000   0.0   3.000000     4.47000      0.0   
50%    2012.000000   0.0      1.730000   0.0   3.240000     6.33000      0.0   
75%    2018.000000   0.0      2.0

In [19]:
# Check time range of the datasets
print("Demand Time Range:", demand_df['Year'].min(), "to", demand_df['Year'].max())
print("Generation Time Range:", generation_df['Year'].min(), "to", generation_df['Year'].max())
print("Per Capita Time Range:", percapita_df['Year'].min(), "to", percapita_df['Year'].max())
print("Population Time Range:", population_df['Year'].min(), "to", population_df['Year'].max())


Demand Time Range: 2000 to 2024
Generation Time Range: 2000 to 2024
Per Capita Time Range: 2000 to 2024
Population Time Range: 2000 to 2024


In [20]:
# Sort the datasets by Year
demand_df = demand_df.sort_values(by='Year')
generation_df = generation_df.sort_values(by='Year')
percapita_df = percapita_df.sort_values(by='Year')
population_df = population_df.sort_values(by='Year')


In [21]:
# Set the 'Year' column as the index for time series analysis
demand_df.set_index('Year', inplace=True)
generation_df.set_index('Year', inplace=True)
percapita_df.set_index('Year', inplace=True)
population_df.set_index('Year', inplace=True)


In [22]:
# Preview the final cleaned datasets
print("Final Demand Dataset")
display(demand_df.head())
print("Final Generation Dataset")
display(generation_df.head())
print("Final Per Capita Dataset")
display(percapita_df.head())
print("Final Population Dataset")
display(population_df.head())


Final Demand Dataset


,Country,Code,Electricity-demand
Year,,,
2000,Kenya,KEN,4.51
2001,Kenya,KEN,4.98
2002,Kenya,KEN,5.37
2003,Kenya,KEN,5.67
2004,Kenya,KEN,6.32


Final Generation Dataset


,Country,Code,Coal,Fossil-fuels,Gas,Hydro,Low-carbon,Nuclear,Oil,Renewables,Solar,Solar-wind,Wind,Geothermal,Total-electricity-produced
Year,,,,,,,,,,,,,,,
2000,Kenya,KEN,0.0,2.57,0.0,1.31,1.74,0.0,2.57,1.74,0.0,0.0,0.0,0.43,4.31
2001,Kenya,KEN,0.0,1.95,0.0,2.38,2.86,0.0,1.95,2.86,0.0,0.0,0.0,0.48,4.81
2002,Kenya,KEN,0.0,1.20,0.0,3.09,3.95,0.0,1.20,3.95,0.0,0.0,0.0,0.86,5.15
2003,Kenya,KEN,0.0,0.99,0.0,3.23,4.49,0.0,0.99,4.49,0.0,0.0,0.0,1.26,5.48
2004,Kenya,KEN,0.0,1.82,0.0,2.84,4.34,0.0,1.82,4.34,0.0,0.0,0.0,1.50,6.16


Final Per Capita Dataset


,Country,Code,Coal-per-capita,Fossil-fuels-per-capita,Gas-per-capita,Hydro-per-capita,Low-carbon-per-capita,Nuclear-per-capita,Oil-per-capita,Renewable-per-capita,Solar-per-capita,Solar-wind-per-capita,Wind-per-capita,Geothermal-per-capita,Total-electricity-per-capita-produced
Year,,,,,,,,,,,,,,,
2000,Kenya,KEN,0.0,83.869360,0.0,42.750530,56.783150,0.0,83.869360,56.783150,0.0,0.0,0.0,14.03262,140.652510
2001,Kenya,KEN,0.0,61.671448,0.0,75.270800,90.451454,0.0,61.671448,90.451454,0.0,0.0,0.0,15.18065,152.122902
2002,Kenya,KEN,0.0,36.776188,0.0,94.698685,121.054955,0.0,36.776188,121.054955,0.0,0.0,0.0,26.35627,157.831143
2003,Kenya,KEN,0.0,29.418552,0.0,95.981740,133.423540,0.0,29.418552,133.423540,0.0,0.0,0.0,37.44180,162.842092
2004,Kenya,KEN,0.0,52.429240,0.0,81.812660,125.023580,0.0,52.429240,125.023580,0.0,0.0,0.0,43.21092,177.452820


Final Population Dataset


,Country,Population
Year,,
2000,Kenya,30642894
2001,Kenya,31619171
2002,Kenya,32629809
2003,Kenya,33652233
2004,Kenya,34713452
